In [4]:
import requests
import pandas as pd

def fetch_solar_and_cloud(lat: float, lon: float, timezone="Asia/Shanghai"):
    """
    获取当前的太阳辐射强度和云层覆盖率
    无需 API Key
    
    参数:
        lat: 纬度
        lon: 经度
        timezone: 时区（宁波使用 Asia/Shanghai）
    
    返回:
        dict: {
            'current_radiation': 当前短波辐射 (W/m²),
            'current_cloudcover': 当前云层覆盖率 (%),
            'timestamp': 数据时间戳
        }
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "shortwave_radiation,cloudcover",  # 太阳辐射 + 云量
        "forecast_days": 1,       # 预报1天
        "timezone": timezone,
    }
    
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    js = r.json()
    
    # 提取最近一小时的数据（假设当前时间对应最新的数据点）
    times = js["hourly"]["time"]
    radiations = js["hourly"]["shortwave_radiation"]  # 单位: W/m²
    cloudcovers = js["hourly"]["cloudcover"]          # 单位: %
    
    # 找到当前时刻对应的索引（取最新）
    latest_idx = len(times) - 1
    
    return {
        'current_radiation': radiations[latest_idx],
        'current_cloudcover': cloudcovers[latest_idx],
        'timestamp': times[latest_idx],
        'all_times': pd.to_datetime(times),
        'all_radiations': radiations,
        'all_cloudcovers': cloudcovers
    }


def fetch_forecast_radiation_and_cloud(lat: float, lon: float, hours=24, timezone="Asia/Shanghai"):
    """
    获取未来 N 小时的太阳辐射强度和云量预报
    用于光伏预测模型输入
    
    参数:
        lat: 纬度
        lon: 经度
        hours: 预报小时数（默认24小时）
        timezone: 时区
    
    返回:
        pd.DataFrame: 包含 time, radiation, cloudcover 三列
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "minutely_15": "shortwave_radiation,cloudcover",
        "forecast_days": 2,       # 预报2天确保有足够数据
        "timezone": timezone,
    }
    
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    js = r.json()
    
    times = js["minutely_15"]["time"]
    radiations = js["minutely_15"]["shortwave_radiation"]
    cloudcovers = js["minutely_15"]["cloudcover"]
    
    # 转为 DataFrame
    df = pd.DataFrame({
        "time": pd.to_datetime(times),
        "radiation_wpm2": radiations,   # 太阳辐射 W/m²
        "cloudcover_pct": cloudcovers   # 云量百分比 0-100
    })
    
    # # 只保留未来 hours 小时（从当前时间开始）
    # now = pd.Timestamp.now(tz=timezone)
    # df = df[df["time"] >= now].head(hours)
    # 只保留未来 hours 小时（从当前时间开始）
    now = pd.Timestamp.now(tz=timezone)
    # 确保时间列也具有时区信息
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert(timezone)
    df = df[df["time"] >= now].head(hours)

    
    return df.reset_index(drop=True)


# ========== 测试示例 ==========
if __name__ == "__main__":
    NINGBO_LAT = 29.8683
    NINGBO_LON = 121.5440
    
    # 获取当前实时数据
    current = fetch_solar_and_cloud(NINGBO_LAT, NINGBO_LON)
    print(f"当前时间: {current['timestamp']}")
    print(f"太阳辐射强度: {current['current_radiation']} W/m²")
    print(f"云层覆盖率: {current['current_cloudcover']}%")
    
    print("\n---")
    
    # 获取未来24小时预报
    forecast = fetch_forecast_radiation_and_cloud(NINGBO_LAT, NINGBO_LON, hours=24)
    print(f"未来 {len(forecast)} 小时预报:")
    print(forecast.head(10))

当前时间: 2026-04-24T23:00
太阳辐射强度: 0.0 W/m²
云层覆盖率: 90%

---
未来 24 小时预报:
                       time  radiation_wpm2  cloudcover_pct
0 2026-04-24 10:45:00+08:00             0.0             100
1 2026-04-24 11:00:00+08:00             0.0             100
2 2026-04-24 11:15:00+08:00             0.0             100
3 2026-04-24 11:30:00+08:00             0.0             100
4 2026-04-24 11:45:00+08:00             0.0             100
5 2026-04-24 12:00:00+08:00             0.0             100
6 2026-04-24 12:15:00+08:00             0.0             100
7 2026-04-24 12:30:00+08:00             0.0              99
8 2026-04-24 12:45:00+08:00             0.0              99
9 2026-04-24 13:00:00+08:00             0.0              98
